# Introducere în Ingineria Prompturilor
Ingineria prompturilor este procesul de proiectare și optimizare a prompturilor pentru sarcini de procesare a limbajului natural. Acesta implică selectarea prompturilor potrivite, ajustarea parametrilor acestora și evaluarea performanței lor. Ingineria prompturilor este crucială pentru atingerea unei acurateți și eficiențe ridicate în modelele NLP. În această secțiune, vom explora bazele ingineriei prompturilor folosind modelele OpenAI pentru explorare.


### Exercițiul 1: Tokenizare
Explorează Tokenizarea folosind tiktoken, un tokenizer rapid open-source de la OpenAI
Vezi [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) pentru mai multe exemple.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Exercițiul 2: Verificarea Configurării Cheii pentru Modelele Microsoft Foundry  

> **Notă:** GitHub Models se va întrerupe la sfârșitul lunii iulie 2026. Acest exercițiu folosește în schimb [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), care oferă același catalog de modele cu încercare gratuită și experiența Azure AI Inference SDK.  

Rulează codul de mai jos pentru a verifica dacă punctul tău final Microsoft Foundry Models este configurat corect. Codul încearcă doar un prompt simplu de bază și validează completarea. Input-ul `oh say can you see` ar trebui să se completeze cam astfel: `by the dawn's early light..`  


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Exercițiul 3: Fictive
Explorează ce se întâmplă când ceri LLM să întoarcă completări pentru un prompt despre un subiect care s-ar putea să nu existe, sau despre subiecte despre care este posibil să nu știe deoarece au fost în afara setului său de date pre-antrenat (mai recent). Vezi cum se schimbă răspunsul dacă încerci un prompt diferit sau un model diferit.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### Exercițiul 4: Bazat pe instrucțiuni 
Folosește variabila "text" pentru a seta conținutul principal 
și variabila "prompt" pentru a oferi o instrucțiune legată de acel conținut principal.

Aici îi cerem modelului să rezume textul pentru un elev de clasa a doua


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### Exercițiul 5: Prompt complex 
Încearcă o solicitare care conține mesaje de sistem, utilizator și asistent 
Sistemul setează contextul asistentului
Mesajele utilizatorului și asistentului oferă contextul unei conversații cu mai multe schimburi

Observă cum personalitatea asistentului este setată pe „sarcastică” în contextul sistemului. 
Încearcă să folosești un context de personalitate diferit. Sau încearcă o serie diferită de mesaje de intrare/ieșire


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Exercițiu: Explorează-ți intuiția
Exemplele de mai sus îți oferă modele pe care le poți folosi pentru a crea noi solicitări (simple, complexe, instrucțiuni etc.) - încearcă să creezi alte exerciții pentru a explora unele dintre celelalte idei despre care am vorbit, cum ar fi exemple, indicii și altele.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
